In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import gc
import os
import time
from tqdm import tqdm

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             confusion_matrix, f1_score,
                             precision_score, recall_score,
                             roc_auc_score, average_precision_score)

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
print("Knižnice načítané")


In [ ]:
# ═══════════════════════════════════════════════════
# KONFIGURÁCIA
# ═══════════════════════════════════════════════════
DATA_PATH = '../Data/Export_data/forecast_1h.parquet'
OUTPUT_DIR = '../Modely/SARIMA/sarima_1h'
os.makedirs(OUTPUT_DIR, exist_ok=True)

VOLTAGE_COLS = ['u1_norm', 'u2_norm', 'u3_norm']
CURRENT_COLS = ['i1_norm', 'i2_norm', 'i3_norm']
TARGET_COLS = VOLTAGE_COLS + CURRENT_COLS

SELECTED_HORIZONS = [1, 3, 6, 12]

TRAINING_WINDOW = 168
FORECAST_HORIZON = 12
MIN_SEGMENT_LENGTH = TRAINING_WINDOW + FORECAST_HORIZON
SEASONAL_PERIOD = 24

# Kandidáti SARIMA - skúsia sa všetci a nakoniec sa použije ten s najnižším AIC.
# Optuna potom nájde ešte lepšiu kombináciu (nižšie v notebooku).
SARIMA_CANDIDATES = [
    ((1, 1, 1), (1, 1, 1, SEASONAL_PERIOD)),
    ((1, 0, 1), (1, 0, 1, SEASONAL_PERIOD)),
    ((2, 1, 1), (1, 1, 0, SEASONAL_PERIOD)),
    ((1, 1, 0), (1, 1, 0, SEASONAL_PERIOD)),
    ((0, 1, 1), (0, 1, 1, SEASONAL_PERIOD)),
]

OPTUNA_TRIALS = 20
OPTUNA_SAMPLE_SEGS = 30
ANOMALY_SIGMA = 3.0   # prah pre anomaly detection (mean + 3σ z čistých)

print(f"Trénovacie okno: {TRAINING_WINDOW} krokov")
print(f"Forecast horizont: {FORECAST_HORIZON} krokov")
print(f"Sezónna perióda: {SEASONAL_PERIOD}")
print(f"Horizonty: {SELECTED_HORIZONS}")


---
# ČASŤ 1: NAČÍTANIE DÁT A ROZDELENIE
---

In [ ]:
%%time
data = pd.read_parquet(DATA_PATH)
data['t_utc'] = pd.to_datetime(data['t_utc'])
# float32 namiesto float64 šetrí pamäť na polovicu
for col in data.select_dtypes(include=['float64']).columns:
    data[col] = data[col].astype('float32')

print(f"Celkovo: {len(data):,} riadkov")
print(f"Chyba=0: {(data['Chyba']==0).sum():,}, Chyba=1: {(data['Chyba']==1).sum():,}")

# Pre každý segment zistíme jeho stav (chyba_max=0 znamená, že celý segment je čistý)
seg_info = data.groupby(['eic', 'segment_id'], observed=True).agg(
    start_time=('t_utc', 'min'), end_time=('t_utc', 'max'),
    num_records=('t_utc', 'count'), chyba_max=('Chyba', 'max')).reset_index()

# Rozdelenie na čisté a poruchové segmenty - sortujeme podľa času pre chronologický split
clean_segs = seg_info[seg_info['chyba_max'] == 0].sort_values('start_time').reset_index(drop=True)
error_segs = seg_info[seg_info['chyba_max'] == 1].sort_values('start_time').reset_index(drop=True)
print(f"\n{len(clean_segs):,} čistých, {len(error_segs):,} poruchových segmentov")

# Chronologický split čistých segmentov 70/15/15 (train/val/test)
# Keďže sú zoradené podľa času, train sú najstaršie a test najnovšie
n_clean = len(clean_segs)
cut_70 = int(n_clean * 0.70)
cut_85 = int(n_clean * 0.85)

# Sety dvojíc (eic, segment_id) - rýchly lookup pri filtrovaní dát nižšie
train_seg_keys = set(zip(clean_segs.iloc[:cut_70]['eic'], clean_segs.iloc[:cut_70]['segment_id']))
val_seg_keys = set(zip(clean_segs.iloc[cut_70:cut_85]['eic'], clean_segs.iloc[cut_70:cut_85]['segment_id']))
test_seg_keys = set(zip(clean_segs.iloc[cut_85:]['eic'], clean_segs.iloc[cut_85:]['segment_id']))
error_seg_keys = set(zip(error_segs['eic'], error_segs['segment_id']))

print(f"Chyba=0: train={len(train_seg_keys)}, val={len(val_seg_keys)}, test={len(test_seg_keys)}")
print(f"Chyba=1: error={len(error_seg_keys)}")

# Filtrovanie dát podľa setov - pre každý riadok overíme, či jeho (eic, segment_id) je v sete
seg_key = list(zip(data['eic'], data['segment_id']))
train_clean = data[np.array([k in train_seg_keys for k in seg_key])].copy()
val_clean = data[np.array([k in val_seg_keys for k in seg_key])].copy()
test_clean = data[np.array([k in test_seg_keys for k in seg_key])].copy()
error_data = data[np.array([k in error_seg_keys for k in seg_key])].copy()
# anomaly_data = test_clean (Chyba=0) + error_data (Chyba=1) - na test anomaly detection
anomaly_data = pd.concat([test_clean, error_data], ignore_index=True)

del seg_key, data; gc.collect()

# Súhrnná tabuľka rozdelenia
sep = '=' * 70
print(f"\n{sep}")
print(f"{'MNOŽINA':<22s} {'RIADKOV':>12s} {'Chyba=0':>12s} {'Chyba=1':>12s} {'Segmentov':>10s}")
print(sep)
for nm, df in [('train_clean', train_clean), ('val_clean', val_clean),
               ('test_clean', test_clean), ('error_data', error_data), ('anomaly_data', anomaly_data)]:
    c0 = (df['Chyba']==0).sum(); c1 = (df['Chyba']==1).sum()
    n_seg = df.groupby(['eic','segment_id'], observed=True).ngroups
    print(f"  {nm:<20s} {len(df):>10,} {c0:>10,} {c1:>10,} {n_seg:>9,}")
print(sep)


---
# ČASŤ 2: SARIMA FUNKCIE
---

In [ ]:
def fit_sarima_auto(series, horizon, candidates):
    # Skúša postupne všetkých kandidátov a vráti predikciu modelu s najnižším AIC.
    # Vracia None, ak sa žiadny model nepodaril fitnúť alebo predikcia vyšla nezmyselná.
    
    # Príliš veľa NaN = nemá zmysel fitovať
    if series.isna().sum() > len(series) * 0.1:
        return None
    # Vyplnenie zvyšných NaN, konverzia na float64 (SARIMAX vyžaduje)
    series_clean = series.ffill().bfill().astype('float64')
    # Konštantný rad - SARIMA zlyhá, lebo nie je čo modelovať
    if series_clean.std() < 1e-10:
        return None
    
    # Štatistiky série - použijú sa na sanity check predikcie
    s_min, s_max = series_clean.min(), series_clean.max()
    s_range = s_max - s_min
    s_mean = series_clean.mean()
    
    best_aic = np.inf
    best_forecast = None
    
    # Vyskúšame všetkých kandidátov a odložíme najlepší (podľa AIC)
    for order, seasonal_order in candidates:
        try:
            # Parametre nastavené pre rýchlosť: powell solver, max 50 iterácií, low_memory mode
            model = SARIMAX(series_clean, order=order, seasonal_order=seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False,
                            simple_differencing=True, concentrate_scale=True)
            fitted = model.fit(disp=False, maxiter=50, method='powell', low_memory=True)
            forecast = fitted.forecast(steps=horizon).values
            
            # Ak vyšli NaN/Inf, kandidát preč
            if np.any(np.isnan(forecast)) or np.any(np.isinf(forecast)):
                continue
            
            # Ak je predikcia ďaleko od pozorovaného rozsahu, orežeme ju (SARIMA niekedy diverguje)
            margin = max(s_range * 3, abs(s_mean) * 0.5, 1.0)
            if np.any(forecast < s_min - margin) or np.any(forecast > s_max + margin):
                forecast = np.clip(forecast, s_min - margin, s_max + margin)
            
            # Najlepší doteraz - odložíme
            if fitted.aic < best_aic:
                best_aic = fitted.aic
                best_forecast = forecast
        except:
            continue
    
    if best_forecast is None:
        return None
    
    # Finálne orezanie a sanity check - ak je priemer predikcie veľmi mimo, zahodíme ju
    margin = max(s_range * 2, abs(s_mean) * 0.3, 1.0)
    best_forecast = np.clip(best_forecast, s_min - margin, s_max + margin)
    if abs(np.mean(best_forecast) - s_mean) > max(s_range * 2, abs(s_mean) * 0.5):
        return None
    return best_forecast


In [ ]:
def run_sarima_on_split(df_split, split_name):
    # Spustí SARIMA na každý segment v danom splite a zozbiera predikcie.
    # Pre každý segment trénujeme na prvom okne (TRAINING_WINDOW) a predikujeme nasledujúci
    # úsek (FORECAST_HORIZON krokov).
    
    result_batches = []
    segs = df_split.groupby(['eic', 'segment_id'], observed=True)
    total = len(segs)
    processed, skipped, failed = 0, 0, 0
    t_start = time.time()
    
    for (eic, seg_id), seg_data in tqdm(segs, desc=f'{split_name}', total=total):
        seg_data = seg_data.sort_values('t_utc').reset_index(drop=True)
        
        # Príliš krátky segment - preskočíme
        if len(seg_data) < MIN_SEGMENT_LENGTH:
            skipped += 1
            continue
        
        # Trénovacie okno = prvých TRAINING_WINDOW krokov, test = ďalších FORECAST_HORIZON
        train_end = TRAINING_WINDOW
        test_end = train_end + FORECAST_HORIZON
        if test_end > len(seg_data):
            skipped += 1
            continue
        
        test_df = seg_data.iloc[train_end:test_end]
        times = test_df['t_utc'].values
        chyba = test_df['Chyba'].values
        actual = test_df[TARGET_COLS].values
        
        seg_failed = False
        # Pre každú zo 6 cieľových premenných (3 napätia + 3 prúdy) trénujeme samostatný SARIMA
        for t_idx, target in enumerate(TARGET_COLS):
            preds = fit_sarima_auto(seg_data[target].iloc[:train_end], FORECAST_HORIZON, SARIMA_CANDIDATES)
            if preds is None:
                seg_failed = True
                break
            
            # Uložíme len vybrané horizonty (napr. h=1, 3, 6, 12 namiesto všetkých)
            for h_idx in range(FORECAST_HORIZON):
                h = h_idx + 1
                if h not in SELECTED_HORIZONS:
                    continue
                result_batches.append({
                    'eic': eic, 'segment_id': seg_id, 't_utc': times[h_idx],
                    'Chyba': int(chyba[h_idx]), 'target': target, 'hour_ahead': h,
                    'prediction': float(preds[h_idx]), 'actual': float(actual[h_idx, t_idx]),
                    'split': split_name})
        
        if seg_failed: failed += 1
        else: processed += 1
    
    elapsed = time.time() - t_start
    print(f"  {split_name}: {processed} OK, {skipped} skip, {failed} fail | {elapsed/60:.1f} min")
    return pd.DataFrame(result_batches)


---
# ČASŤ 3: OPTUNA TUNING
---

In [ ]:
%%time
print("=" * 60)
print("OPTUNA HYPERPARAMETER TUNING (SARIMA)")
print("=" * 60)

# Pre tuning si vyberieme náhodnú podmnožinu validačných segmentov - tunovať na všetkých
# by trvalo príliš dlho. seed=42 zabezpečí, že ten istý sample dostaneme aj pri opätovnom spustení.
val_segs = val_clean.groupby(['eic', 'segment_id'], observed=True)
seg_keys = list(val_segs.groups.keys())
np.random.seed(42)
sample_keys = [seg_keys[i] for i in np.random.choice(len(seg_keys),
               min(OPTUNA_SAMPLE_SEGS, len(seg_keys)), replace=False)]
print(f"Vzorka: {len(sample_keys)} segmentov z {len(seg_keys)}")

def sarima_objective(trial):
    # Optuna navrhne hodnoty parametrov, my ich otestujeme na sample segmentov
    # a vrátime priemernú MAE - Optuna sa snaží toto minimalizovať
    p = trial.suggest_int('p', 0, 3)
    d = trial.suggest_int('d', 0, 2)
    q = trial.suggest_int('q', 0, 3)
    P = trial.suggest_int('P', 0, 2)
    D = trial.suggest_int('D', 0, 1)
    Q = trial.suggest_int('Q', 0, 2)
    tw = trial.suggest_int('training_window', 72, TRAINING_WINDOW, step=24)
    
    min_len = tw + FORECAST_HORIZON
    order = (p, d, q)
    seasonal_order = (P, D, Q, SEASONAL_PERIOD)
    all_errors = []
    
    for (eic, seg_id) in sample_keys:
        seg_data = val_segs.get_group((eic, seg_id)).sort_values('t_utc').reset_index(drop=True)
        if len(seg_data) < min_len: continue
        
        actual = seg_data[TARGET_COLS].iloc[tw:tw+FORECAST_HORIZON].values
        seg_errors = []
        
        # Pre rýchlosť tunujeme len na napätiach (TARGET_COLS[:3]) a nie na všetkých 6 stĺpcoch
        for t_idx, target in enumerate(TARGET_COLS[:3]):
            try:
                series = seg_data[target].iloc[:tw].ffill().bfill()
                if series.std() < 0.01: continue
                model = SARIMAX(series, order=order, seasonal_order=seasonal_order,
                                enforce_stationarity=False, enforce_invertibility=False,
                                simple_differencing=True, concentrate_scale=True)
                fitted = model.fit(disp=False, maxiter=50, method='powell', low_memory=True)
                forecast = fitted.forecast(steps=FORECAST_HORIZON).values
                if not np.any(np.isnan(forecast)):
                    seg_errors.append(np.abs(actual[:, t_idx] - forecast).mean())
            except: continue
        
        if seg_errors: all_errors.append(np.mean(seg_errors))
    
    # Ak sa nepodarilo nič fitnúť, vrátime nekonečno (Optuna túto kombináciu zahodí)
    if len(all_errors) == 0: return float('inf')
    return np.mean(all_errors)

study = optuna.create_study(direction='minimize', study_name='sarima_tuning')
study.optimize(sarima_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

print(f"\nNajlepšie parametre:")
for k, v in study.best_params.items(): print(f"  {k}: {v}")
print(f"Najlepšia MAE: {study.best_value:.6f}")

# Najlepšiu kombináciu nastavíme ako jediného kandidáta pre finálne trénovanie
bp = study.best_params
SARIMA_CANDIDATES = [((bp['p'], bp['d'], bp['q']), (bp['P'], bp['D'], bp['Q'], SEASONAL_PERIOD))]
TRAINING_WINDOW = bp['training_window']
MIN_SEGMENT_LENGTH = TRAINING_WINDOW + FORECAST_HORIZON
print(f"\nSARIMA order: {SARIMA_CANDIDATES[0][0]} × {SARIMA_CANDIDATES[0][1]}")
print(f"TRAINING_WINDOW: {TRAINING_WINDOW}")


---
# ČASŤ 4: TRÉNOVANIE A FORECASTING
---

In [ ]:
%%time
print("=" * 60)
print("SARIMA - FORECASTING (len Chyba=0)")
print("=" * 60)

# Forecasting na čistých dátach - validácia + test
print("\n--- VALIDÁCIA ---")
val_results = run_sarima_on_split(val_clean, 'val')
del val_clean; gc.collect()

print("\n--- TEST ---")
test_results = run_sarima_on_split(test_clean, 'test')

# Spojenie do jedného DataFrame + pridanie reziduí (skutočné - predikované)
forecast_df = pd.concat([val_results, test_results], ignore_index=True)
if len(forecast_df) > 0:
    forecast_df['residual'] = forecast_df['actual'] - forecast_df['prediction']
    forecast_df['abs_error'] = np.abs(forecast_df['residual'])
del val_results, test_results; gc.collect()
print(f"\nCelkovo predikcií (čisté): {len(forecast_df):,}")


In [ ]:
# Metriky pre validáciu aj test, oddelene pre napätie a prúd
for split in ['val', 'test']:
    split_label = 'VALIDATION (Chyba=0)' if split == 'val' else 'TEST (Chyba=0)'
    sd = forecast_df[forecast_df['split'] == split]
    print(f"\n{'='*60}")
    print(f"  {split_label}: {len(sd):,} predikcií")
    print(f"{'='*60}")
    
    # Napätie
    print(f"\n  NAPÄTIE:")
    rows_v = []
    for t in VOLTAGE_COLS:
        d = sd[sd['target'] == t].dropna(subset=['actual', 'prediction'])
        if len(d) < 2: continue
        rows_v.append({'target': t, 'MAE': mean_absolute_error(d['actual'], d['prediction']),
            'RMSE': np.sqrt(mean_squared_error(d['actual'], d['prediction'])), 'R2': r2_score(d['actual'], d['prediction'])})
    if rows_v:
        mv = pd.DataFrame(rows_v); print(mv.to_string(index=False))
        print(f"    → Priemer: MAE={mv['MAE'].mean():.4f}, RMSE={mv['RMSE'].mean():.4f}, R²={mv['R2'].mean():.4f}")
    
    # Prúd
    print(f"\n  PRÚDY:")
    rows_c = []
    for t in CURRENT_COLS:
        d = sd[sd['target'] == t].dropna(subset=['actual', 'prediction'])
        if len(d) < 2: continue
        rows_c.append({'target': t, 'MAE': mean_absolute_error(d['actual'], d['prediction']),
            'RMSE': np.sqrt(mean_squared_error(d['actual'], d['prediction'])), 'R2': r2_score(d['actual'], d['prediction'])})
    if rows_c:
        mc = pd.DataFrame(rows_c); print(mc.to_string(index=False))
        print(f"    → Priemer: MAE={mc['MAE'].mean():.4f}, RMSE={mc['RMSE'].mean():.4f}, R²={mc['R2'].mean():.4f}")


In [ ]:
# Ako klesá presnosť so vzdialenosťou predikcie - h1 by mal byť najpresnejší, h12 najmenej
print("METRIKY PODĽA HORIZONTU (test, Chyba=0):")
sd = forecast_df[forecast_df['split'] == 'test']

print("\n  Napätie:")
for h in SELECTED_HORIZONS:
    d = sd[(sd['hour_ahead']==h) & (sd['target'].isin(VOLTAGE_COLS))]
    if len(d) < 2: continue
    print(f"    h{h:2d}: MAE={mean_absolute_error(d['actual'], d['prediction']):.4f}, "
          f"RMSE={np.sqrt(mean_squared_error(d['actual'], d['prediction'])):.4f}, "
          f"R²={r2_score(d['actual'], d['prediction']):.4f}")

print("\n  Prúdy:")
for h in SELECTED_HORIZONS:
    d = sd[(sd['hour_ahead']==h) & (sd['target'].isin(CURRENT_COLS))]
    if len(d) < 2: continue
    print(f"    h{h:2d}: MAE={mean_absolute_error(d['actual'], d['prediction']):.4f}, "
          f"RMSE={np.sqrt(mean_squared_error(d['actual'], d['prediction'])):.4f}, "
          f"R²={r2_score(d['actual'], d['prediction']):.4f}")


---
# ČASŤ 5: ANOMALY DETECTION
---

In [ ]:
%%time
print("=" * 60)
print("ANOMALY DETECTION (SARIMA na zmiešaných dátach)")
print("=" * 60)

# Spustenie SARIMA na anomaly_data (test_clean + error_data) - rovnaký princíp ako pri
# forecastingu, ale teraz nás zaujímajú reziduí ako anomaly score
anomaly_results = run_sarima_on_split(anomaly_data, 'anomaly')
del anomaly_data, error_data; gc.collect()

if len(anomaly_results) > 0:
    anomaly_results['residual'] = anomaly_results['actual'] - anomaly_results['prediction']
    anomaly_results['abs_error'] = np.abs(anomaly_results['residual'])

    # Anomaly score = priemerná absolútna chyba cez všetky cieľové premenné v danom čase
    # Vyšší score = predikcia sa viac líši od skutočnosti = pravdepodobnejšia anomália
    anom_agg = anomaly_results.groupby(['eic', 'segment_id', 't_utc', 'hour_ahead'], observed=True).agg(
        Chyba=('Chyba', 'first'), anomaly_score=('abs_error', 'mean')).reset_index()

    print(f"\nanom_agg: {len(anom_agg):,} (Chyba=0: {(anom_agg['Chyba']==0).sum():,}, Chyba=1: {(anom_agg['Chyba']==1).sum():,})")

    # Prah nastavíme na mean + 3σ z čistých dát - štandardná IQR-podobná hranica.
    # Všetko nad prahom označíme ako anomáliu
    clean_scores = anom_agg[anom_agg['Chyba'] == 0]['anomaly_score'].values
    threshold = clean_scores.mean() + ANOMALY_SIGMA * clean_scores.std()
    print(f"\nPrah (mean + {ANOMALY_SIGMA}σ z čistých): {threshold:.4f}")
    print(f"  mean = {clean_scores.mean():.4f}, std = {clean_scores.std():.4f}")

    anom_agg['predicted_anomaly'] = (anom_agg['anomaly_score'] > threshold).astype(int)
    y_true = anom_agg['Chyba'].values; y_pred = anom_agg['predicted_anomaly'].values
    scores = anom_agg['anomaly_score'].values

    # Confusion matrix - .ravel() rozbalí maticu 2x2 na 4 čísla [tn, fp, fn, tp]
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1]); tn, fp, fn, tp = cm.ravel()
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\nCONFUSION MATRIX:")
    print(f"                  Predikované")
    print(f"                Normal  Anomália")
    print(f"  Skut. Normal  {tn:>7,}  {fp:>7,}")
    print(f"  Skut. Chyba   {fn:>7,}  {tp:>7,}")
    print(f"\n  Precision: {prec:.4f}\n  Recall:    {rec:.4f}\n  F1-score:  {f1:.4f}")
    
    # ROC-AUC a PR-AUC nepotrebujú prah - pracujú priamo s anomaly_score
    try:
        print(f"  ROC-AUC:   {roc_auc_score(y_true, scores):.4f}")
        print(f"  PR-AUC:    {average_precision_score(y_true, scores):.4f}")
    except: pass

    # Metriky rozdelené po horizontoch - či sa anomaly detection zhoršuje pre dlhšie predikcie
    print(f"\nPODĽA HORIZONTU:")
    for h in SELECTED_HORIZONS:
        hd = anom_agg[anom_agg['hour_ahead']==h]
        if len(hd) == 0: continue
        print(f"  h{h:2d}: P={precision_score(hd['Chyba'], hd['predicted_anomaly'], zero_division=0):.4f}, "
              f"R={recall_score(hd['Chyba'], hd['predicted_anomaly'], zero_division=0):.4f}, "
              f"F1={f1_score(hd['Chyba'], hd['predicted_anomaly'], zero_division=0):.4f}")
else:
    print("Žiadne úspešné predikcie")


In [ ]:
# Citlivostná analýza prahu - skúsime rôzne σ a pozrieme, ako sa menia precision/recall.
# Pomáha vidieť trade-off: nižšie σ = viac true positives ale aj viac false positives.
if len(anomaly_results) > 0:
    print("THRESHOLD ANALÝZA:")
    print(f"{'σ':>5s} {'Prec':>8s} {'Recall':>8s} {'F1':>8s} {'TP':>7s} {'FP':>7s} {'FN':>7s}")
    sigma_results = []
    
    for sigma in [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
        th = clean_scores.mean() + sigma * clean_scores.std()
        yp = (scores > th).astype(int)
        sp = precision_score(y_true, yp, zero_division=0)
        sr = recall_score(y_true, yp, zero_division=0)
        sf = f1_score(y_true, yp, zero_division=0)
        s_cm = confusion_matrix(y_true, yp, labels=[0,1]); _, sfp, sfn, stp = s_cm.ravel()
        sigma_results.append({'σ': sigma, 'P': sp, 'R': sr, 'F1': sf})
        print(f"{sigma:5.1f} {sp:8.4f} {sr:8.4f} {sf:8.4f} {stp:7d} {sfp:7d} {sfn:7d}")
    
    # Vykreslenie ako funkcie σ - vidíme, kde sa P, R, F1 "prelomia"
    sr_df = pd.DataFrame(sigma_results)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(sr_df['σ'], sr_df['P'], 'b-o', label='Precision', linewidth=2)
    ax.plot(sr_df['σ'], sr_df['R'], 'r-s', label='Recall', linewidth=2)
    ax.plot(sr_df['σ'], sr_df['F1'], 'g-^', label='F1', linewidth=2)
    ax.set_xlabel('σ'); ax.set_ylabel('Skóre'); ax.set_title('Threshold analýza')
    ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/threshold_analysis.png', dpi=150, bbox_inches='tight'); plt.show()


---
# ČASŤ 6: VIZUALIZÁCIE
---

In [ ]:
# Scatter plot: skutočné vs predikované hodnoty napätia
# Diagonála = perfektná predikcia, body čím bližšie k nej tým lepšie
td = forecast_df[forecast_df['split'] == 'test']
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for i, t in enumerate(VOLTAGE_COLS):
    ax = axes[i]; d = td[td['target'] == t].dropna(subset=['actual', 'prediction'])
    if len(d) < 2: continue
    r2 = r2_score(d['actual'], d['prediction'])
    
    # Pri väčšom počte bodov sample-neme - 5000 bodov stačí na dobrý vizuál a kreslí sa rýchlejšie
    if len(d) > 5000: d = d.sample(5000, random_state=42)
    
    ax.scatter(d['actual'], d['prediction'], alpha=0.12, s=3, c='#5B4F9E')
    # Diagonálna čiara od min do max - ideálna predikcia
    lims = [d[['actual','prediction']].min().min(), d[['actual','prediction']].max().max()]
    ax.plot(lims, lims, 'k--', alpha=0.4)
    ax.set_xlabel('Skutočné [V]'); ax.set_ylabel('Predikované [V]')
    ax.set_title(f'{t}  (R² = {r2:.4f})')

plt.suptitle('Reálne vs Predikované – Napätie (test, Chyba=0)', fontsize=13, y=1.01)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/scatter_voltage.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# To isté pre prúd
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for i, t in enumerate(CURRENT_COLS):
    ax = axes[i]; d = td[td['target'] == t].dropna(subset=['actual', 'prediction'])
    if len(d) < 2: continue
    r2 = r2_score(d['actual'], d['prediction'])
    if len(d) > 5000: d = d.sample(5000, random_state=42)
    
    ax.scatter(d['actual'], d['prediction'], alpha=0.12, s=3, c='#D4582A')
    lims = [d[['actual','prediction']].min().min(), d[['actual','prediction']].max().max()]
    ax.plot(lims, lims, 'k--', alpha=0.4)
    ax.set_xlabel('Skutočné [A]'); ax.set_ylabel('Predikované [A]')
    ax.set_title(f'{t}  (R² = {r2:.4f})')

plt.suptitle('Reálne vs Predikované – Prúdy (test, Chyba=0)', fontsize=13, y=1.01)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/scatter_current.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Časový priebeh anomaly score - vyberáme EIC s najviac poruchovými záznamami,
# aby sme videli, ako reziduály "vystrelia" počas reálnej poruchy
if len(anomaly_results) > 0:
    # Stredný horizont (napr. h=3 z [1,3,6,12]) - kompromis medzi presnosťou a dlhodobosťou
    mid_h = SELECTED_HORIZONS[len(SELECTED_HORIZONS)//2]
    fault_eics = anom_agg[anom_agg['Chyba']==1]['eic'].unique()
    
    if len(fault_eics) > 0:
        # EIC s najviac poruchovými záznamami - tam bude najjasnejší "príbeh"
        best_eic = anom_agg[anom_agg['Chyba']==1].groupby('eic').size().idxmax()
        eic_data = anom_agg[(anom_agg['eic']==best_eic) & (anom_agg['hour_ahead']==mid_h)].sort_values('t_utc')
        
        fig, ax = plt.subplots(figsize=(14, 4))
        times = pd.to_datetime(eic_data['t_utc'])
        ax.plot(times, eic_data['anomaly_score'], color='#333', linewidth=0.6, alpha=0.8)
        
        # Horizontálna čiara = prah anomálie
        ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1.2, label=f'Prah ({ANOMALY_SIGMA}σ)')
        
        # Vyfarbené pásmo = obdobia, kedy bola skutočná porucha
        fault_mask = eic_data['Chyba'].values == 1
        if fault_mask.any():
            ymax = max(eic_data['anomaly_score'].max() * 1.2, threshold * 1.3)
            ax.fill_between(times, 0, ymax, where=fault_mask, alpha=0.10, color='red', label='Skutočná porucha')
        
        ax.set_xlabel('Čas'); ax.set_ylabel('Anomaly score')
        ax.set_title(f'Anomaly score – EIC: {best_eic} (h{mid_h})')
        ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/anomaly_score_time.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Časový priebeh - napätie u1_norm pre náhodný EIC s dobrým R²
mid_h = SELECTED_HORIZONS[0]
td = forecast_df[(forecast_df['split'] == 'test') & (forecast_df['target'] == 'u1_norm') & 
                  (forecast_df['hour_ahead'] == mid_h)].dropna(subset=['actual', 'prediction'])
WINDOW = 24

# Vyberieme len EIC, kde R² je dosť vysoké - ukážka by mala byť reprezentatívna
eic_r2 = {}
for eic, grp in td.groupby('eic'):
    if len(grp) >= 2:
        r2 = r2_score(grp['actual'], grp['prediction'])
        if r2 > 0.85: eic_r2[eic] = r2

if len(eic_r2) > 0:
    # Náhodný výber jedného z dobrých EIC. seed=42 pre reprodukovateľnosť ukážky
    np.random.seed(42)
    selected_eic = np.random.choice(list(eic_r2.keys()))
    eic_data = td[td['eic'] == selected_eic].sort_values('t_utc')
    
    fig, ax = plt.subplots(figsize=(14, 4))
    t = pd.to_datetime(eic_data['t_utc'])
    ax.plot(t, eic_data['actual'], color='#1f77b4', linewidth=1.5, marker='o', markersize=3, label='Skutočné')
    ax.plot(t, eic_data['prediction'], color='#ff7f0e', linewidth=1.5, marker='s', markersize=3, label='Predikované')
    ax.set_ylabel('U1 [V]'); ax.set_xlabel('Čas')
    ax.set_title(f'Predikcia napätia U1 – EIC: {selected_eic} (h{mid_h}, R²={eic_r2[selected_eic]:.4f})')
    ax.legend(); ax.grid(True, alpha=0.3); ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/timeseries_voltage_forecast.png', dpi=150, bbox_inches='tight'); plt.show()
else:
    print("Žiadny EIC s dostatočným R²")


In [ ]:
# To isté pre prúd i1_norm
td_i = forecast_df[(forecast_df['split'] == 'test') & (forecast_df['target'] == 'i1_norm') & 
                    (forecast_df['hour_ahead'] == mid_h)].dropna(subset=['actual', 'prediction'])

eic_r2_i = {}
for eic, grp in td_i.groupby('eic'):
    if len(grp) >= 2:
        r2 = r2_score(grp['actual'], grp['prediction'])
        if r2 > 0.85: eic_r2_i[eic] = r2

if len(eic_r2_i) > 0:
    # Iný seed (123) - aby sa nevybral ten istý EIC ako pri napätí
    np.random.seed(123)
    selected_eic = np.random.choice(list(eic_r2_i.keys()))
    eic_data = td_i[td_i['eic'] == selected_eic].sort_values('t_utc')
    
    fig, ax = plt.subplots(figsize=(14, 4))
    t = pd.to_datetime(eic_data['t_utc'])
    ax.plot(t, eic_data['actual'], color='#1f77b4', linewidth=1.5, marker='o', markersize=3, label='Skutočné')
    ax.plot(t, eic_data['prediction'], color='#ff7f0e', linewidth=1.5, marker='s', markersize=3, label='Predikované')
    ax.set_ylabel('I1 [A]'); ax.set_xlabel('Čas')
    ax.set_title(f'Predikcia prúdu I1 – EIC: {selected_eic} (h{mid_h}, R²={eic_r2_i[selected_eic]:.4f})')
    ax.legend(); ax.grid(True, alpha=0.3); ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/timeseries_current_forecast.png', dpi=150, bbox_inches='tight'); plt.show()
else:
    print("Žiadny EIC s dostatočným R²")


---
# ČASŤ 7: EXPORT VÝSLEDKOV
---

In [ ]:
# Uloženie predikcií do CSV pre ďalšiu analýzu/porovnanie s inými modelmi
forecast_df.to_csv(f'{OUTPUT_DIR}/sarima_forecast_results.csv', sep=';', index=False)
if len(anomaly_results) > 0:
    anom_agg.to_csv(f'{OUTPUT_DIR}/sarima_anomaly_results.csv', sep=';', index=False)

print(f"Uložené do: {OUTPUT_DIR}/")
print(f"  - sarima_forecast_results.csv (forecasting na čistých)")
if len(anomaly_results) > 0:
    print(f"  - sarima_anomaly_results.csv (detekcia na zmiešaných)")
